# Module 0: NumPy Foundations

Welcome! Before we build diffusion models, we need to get really comfortable with NumPy. Every operation in a diffusion pipeline — noise scheduling, the forward process, sampling — boils down to array manipulations. Once you can think fluently in terms of shapes and broadcasting, the later modules will click into place.

Here's what we'll work through together:

- **0.1** Memory layout — strides, views vs. copies, contiguity
- **0.2** Broadcasting — the three rules, common shape patterns
- **0.3** Advanced indexing — fancy indexing, boolean masks, `np.where`
- **0.4** Einsum — a single notation for any tensor contraction
- **0.5** Vectorization — replacing Python loops with NumPy ops (10–100x speedup)
- **0.6** Random number generation — the reparameterization trick

**Prerequisites:** Python fluency, basic linear algebra, introductory probability/statistics.

Each section has a short explanation, worked examples, and exercises. At the end, five capstone challenges tie everything together. Let's get started.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import Tuple

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(42)
print(f"NumPy version: {np.__version__}")

NumPy version: 2.4.2


---
## 0.1 — Memory Layout: Strides, Views, and Contiguity

Understanding how NumPy stores data in memory helps you avoid two common problems: silent bugs from unintended aliasing, and unnecessary memory copies that slow things down.

### ndarray internals

Under the hood, every NumPy array is just a thin wrapper around a flat buffer of bytes. Three pieces of metadata tell NumPy how to interpret that buffer as a multi-dimensional grid:

- **`shape`** — the dimensions of the array (e.g. `(3, 4)` for 3 rows, 4 columns)
- **`strides`** — how many bytes to skip to move one step along each axis
- **`dtype`** — the data type of each element (e.g. `float32` = 4 bytes)

### Strides

Strides are how NumPy navigates a flat chunk of memory as if it were multi-dimensional. They tell you the number of bytes to jump to reach the next element along each axis.

- For a `(3, 4)` float32 array: `strides = (16, 4)`
- Moving one row down = skip 4 floats × 4 bytes = **16 bytes**
- Moving one column right = skip 1 float × 4 bytes = **4 bytes**

### Views vs. copies

This is where bugs hide. Some operations return a **view** (a new array object sharing the same underlying data), while others return a **copy** (a completely independent allocation).

- **View** — slicing: `a[::2]`, `a[:, 1:3]`, `a.reshape(...)`. Mutating the view mutates the original.
- **Copy** — fancy indexing: `a[[0, 2]]`, `a[a > 5]`. Mutating the copy leaves the original untouched.

The worked example below makes this concrete.

### Contiguity

Memory layout determines how efficiently NumPy (and later, PyTorch) can process your data.

- **C-contiguous (row-major)** — rows are stored end-to-end in memory. This is NumPy's default.
- **F-contiguous (column-major)** — columns are stored end-to-end, like Fortran.
- Transpose (`a.T`) just swaps the strides without copying data, so `a.T` is F-contiguous if `a` was C-contiguous.

In [4]:
# --- Worked Example: Inspecting array internals ---

a = np.arange(12, dtype=np.float32).reshape(3, 4)  # shape (3, 4)
print("Array a:")
print(a)
print(f"  dtype   : {a.dtype}")
print(f"  shape   : {a.shape}")
print(f"  strides : {a.strides}  (bytes per axis)")
print(f"  itemsize: {a.itemsize} bytes")
print(f"  C-contig: {a.flags['C_CONTIGUOUS']}")
print(f"  F-contig: {a.flags['F_CONTIGUOUS']}")
print()

# Strides explanation: stride[0]=16 means skip 16 bytes (4 floats * 4 bytes) to move one row down.
# stride[1]=4 means skip 4 bytes (1 float) to move one column right.

# Transpose: strides swap, no data copy
b = a.T  # shape (4, 3)
print("Transposed b = a.T:")
print(f"  shape   : {b.shape}")
print(f"  strides : {b.strides}  (swapped!)")
print(f"  C-contig: {b.flags['C_CONTIGUOUS']}")
print(f"  F-contig: {b.flags['F_CONTIGUOUS']}")
print(f"  Shares memory with a? {np.shares_memory(a, b)}")

Array a:
[[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]]
  dtype   : float32
  shape   : (3, 4)
  strides : (16, 4)  (bytes per axis)
  itemsize: 4 bytes
  C-contig: True
  F-contig: False

Transposed b = a.T:
  shape   : (4, 3)
  strides : (4, 16)  (swapped!)
  C-contig: False
  F-contig: True
  Shares memory with a? True


In [7]:
# --- View vs Copy demo ---

# Slicing returns a VIEW
view_of_a = a[0:2, 1:3]  # shape (2, 2)
print(f"view_of_a shares memory with a? {np.shares_memory(view_of_a, a)}")

view_of_a[0, 0] = -999.0
print(f"After mutating view, a[0,1] = {a[0, 1]}  (mutated through view!)")
a[0, 1] = 1.0  # restore
# Fancy indexing returns a COPY
copy_of_a = a[[0, 2], :]  # shape (2, 4)
print(f"\ncopy_of_a shares memory with a? {np.shares_memory(copy_of_a, a)}")

copy_of_a[0, 0] = -999.0
print(f"After mutating copy, a[0,0] = {a[0, 0]}  (unchanged!)")

view_of_a shares memory with a? True
After mutating view, a[0,1] = -999.0  (mutated through view!)

copy_of_a shares memory with a? False
After mutating copy, a[0,0] = 0.0  (unchanged!)


In [ ]:
# --- dtype performance difference ---

size = 5_000_000
x64 = rng.standard_normal(size)               # float64 by default
x32 = x64.astype(np.float32)                  # float32

t0 = time.perf_counter()
_ = x64 @ x64
t64 = time.perf_counter() - t0

t0 = time.perf_counter()
_ = x32 @ x32
t32 = time.perf_counter() - t0

print(f"float64 dot: {t64*1000:.2f} ms  |  float32 dot: {t32*1000:.2f} ms")
print(f"Memory: float64 = {x64.nbytes / 1e6:.1f} MB, float32 = {x32.nbytes / 1e6:.1f} MB")

### Exercise 0.1

1. Create a `(6, 8)` float64 array `m`. Predict its strides (hint: each float64 is 8 bytes), then verify with `m.strides`.

2. Take a slice `s = m[::2, ::3]`.
   - Is `s` a view or a copy? (Check with `np.shares_memory(m, s)`.)
   - What are its strides? Verify.

3. Call `s_flat = s.reshape(-1)`. Does this return a view or a copy? Why?

4. Write a function `force_c_contiguous(arr)` that returns `arr` unchanged if it is already C-contiguous, otherwise returns a C-contiguous copy.

In [ ]:
# YOUR CODE HERE — Exercise 0.1

# 1. Create a (6, 8) float64 array m. Predict its strides, then verify.
m = np.arange(48, dtype=np.float64).reshape(6, 8)
stride = m.strides
# Prediction: stride[0] = ?, stride[1] = ?
assert stride[0] == 8 * 8 and stride[1] == 8, stride
# print(f"m.strides = {m.strides}")

# 2. Take a slice s = m[::2, ::3]. View or copy? What strides?

# 3. Call s.reshape(-1). View or copy? Why?

# 4. Write force_c_contiguous(arr) that returns arr if C-contiguous,
#    else returns a C-contiguous copy.
def force_c_contiguous(arr: np.ndarray) -> np.ndarray:
    """Return arr if C-contiguous, else a C-contiguous copy."""
    # ===================== YOUR CODE HERE =====================
    pass  # Replace this with your implementation
    # ====================== END YOUR CODE ======================


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# 1. Strides prediction: float64 = 8 bytes, shape (6,8)
#    stride[0] = 8 * 8 = 64, stride[1] = 8
m = np.arange(48, dtype=np.float64).reshape(6, 8)  # shape (6, 8)
print(f"m.strides = {m.strides}")  # (64, 8) -- confirmed

# 2. Slicing with steps returns a VIEW with scaled strides
s = m[::2, ::3]  # shape (3, 3) -- rows 0,2,4; cols 0,3,6
print(f"s is a view: {np.shares_memory(s, m)}")
print(f"s.strides = {s.strides}")  # (128, 24) = (64*2, 8*3)
print(f"s.shape   = {s.shape}")

# 3. reshape(-1) on non-contiguous array requires a COPY
s_flat = s.reshape(-1)  # shape (9,)
print(f"\ns_flat shares memory with m? {np.shares_memory(s_flat, m)}")  # False -- copy!
# Because s has non-unit strides on both axes, the elements are not
# contiguous in memory, so reshape must copy.


# 4.
def force_c_contiguous(arr: np.ndarray) -> np.ndarray:
    """Return arr if C-contiguous, else a C-contiguous copy."""
    if arr.flags['C_CONTIGUOUS']:
        return arr
    return np.ascontiguousarray(arr)


# Test
t = m.T  # F-contiguous, not C-contiguous
t_c = force_c_contiguous(t)
print(f"\nt is C-contig: {t.flags['C_CONTIGUOUS']}")
print(f"t_c is C-contig: {t_c.flags['C_CONTIGUOUS']}")
print(f"t_c shares memory with m: {np.shares_memory(t_c, m)}")
assert np.array_equal(t, t_c)

---
## 0.2 — Broadcasting

Broadcasting is NumPy's mechanism for doing arithmetic on arrays with different shapes — without explicitly copying data. It's the single most important NumPy concept for writing clean, fast code.

### The three rules

NumPy compares shapes element-wise, starting from the **rightmost** (trailing) dimension. Here's how it decides whether two shapes are compatible:

**Rule 1: Pad with 1s.** If the arrays have different numbers of dimensions, the shorter shape gets padded with 1s on the left.

> Example: shapes `(3, 4)` and `(4,)` → the second becomes `(1, 4)`

**Rule 2: Stretch 1s.** Any dimension of size 1 gets "stretched" to match the other array's size along that dimension. (Conceptually — no data is actually copied.)

> Example: `(3, 4)` and `(1, 4)` → the second broadcasts to `(3, 4)`

**Rule 3: Mismatch = error.** If two dimensions disagree and neither is 1, broadcasting fails.

> Example: `(3, 4)` and `(3,)` → trailing dims are 4 vs 3 → **error**

### Walking through an example

Let's trace the rules on `(3, 4) + (4,)`:

```
Step 1 — Right-align the shapes:
    A:  (3, 4)
    b:     (4,)  →  pad with 1  →  (1, 4)

Step 2 — Compare each dimension:
    Dim 0:  3 vs 1  →  stretch 1 to 3  ✓
    Dim 1:  4 vs 4  →  match           ✓

Result shape: (3, 4)
```

Each of the 3 rows of `A` gets `b` added element-wise. No loop, no copy of `b`.

In [ ]:
# --- Worked Example 1: Row + vector → broadcasts across rows ---
A = np.ones((3, 4))    # shape (3, 4)
b = np.array([10, 20, 30, 40])  # shape (4,)

# Right-align:  (3, 4)
#                  (4,)  → treated as (1, 4) → broadcast to (3, 4)
result = A + b  # shape (3, 4)
print("Example 1: (3, 4) + (4,)")
print(result)
print(f"Result shape: {result.shape}\n")

In [ ]:
# --- Worked Example 2: Outer sum via broadcasting ---
x = np.array([1, 2, 3])        # shape (3,)
y = np.array([10, 20, 30, 40]) # shape (4,)

# Reshape x to (3, 1) so broadcasting gives (3, 1) + (4,) → (3, 4)
outer_sum = x[:, None] + y[None, :]  # shape (3, 4)
print("Example 2: Outer sum — (3, 1) + (1, 4) → (3, 4)")
print(outer_sum)
print(f"Result shape: {outer_sum.shape}\n")

# --- Worked Example 3: Per-channel normalization (images) ---
# A batch of 3-channel images: (B, C, H, W)
images = rng.standard_normal((8, 3, 32, 32)).astype(np.float32)

# Channel means/stds have shape (3,) — we need shape (1, 3, 1, 1) for broadcasting
channel_means = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
channel_stds  = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)

# Broadcasting: (8, 3, 32, 32) - (1, 3, 1, 1) → (8, 3, 32, 32)
normalized = (images - channel_means) / channel_stds
print(f"Example 3: Image normalization — {images.shape} - {channel_means.shape} → {normalized.shape}")
print(f"Before: channel means = {images.mean(axis=(0,2,3)).round(3)}")
print(f"After:  channel means = {normalized.mean(axis=(0,2,3)).round(3)}\n")

# --- Worked Example 4: Scaling a batch by per-sample coefficients ---
# In diffusion: alpha_bar has shape (B,) and x_0 has shape (B, C, H, W)
B = 4
alpha_bar = np.array([0.9, 0.5, 0.2, 0.05])  # shape (B,)
x_0 = rng.standard_normal((B, 1, 8, 8)).astype(np.float32)

# Reshape alpha_bar to (B, 1, 1, 1) so it broadcasts across C, H, W
scaled = alpha_bar[:, None, None, None] * x_0  # (B, 1, 1, 1) * (B, 1, 8, 8) → (B, 1, 8, 8)
print(f"Example 4: Per-sample scaling — {alpha_bar[:, None, None, None].shape} * {x_0.shape} → {scaled.shape}")
print(f"alpha_bar = {alpha_bar}")
print(f"Energy ratio per sample: {(scaled**2).mean(axis=(1,2,3)).round(3) / (x_0**2).mean(axis=(1,2,3)).round(3)}")

### Exercise 0.2

1. **Predict the output shape** (or "error") for each pair:
   - `(5, 3) + (3,)`
   - `(5, 3) + (5,)`
   - `(2, 1, 4) + (3, 4)`
   - `(2, 3, 4) + (2, 1, 1)`
   - `(7,) + (1, 7, 1)`

2. **Normalize a batch of images per-channel.** Given `images` of shape `(B, H, W, C)` and per-channel `mean` and `std` each of shape `(C,)`, normalize so each channel has zero mean and unit variance. Think about which dimensions broadcast automatically.

3. **Pairwise Euclidean distances.** Given `X` of shape `(N, D)` and `Y` of shape `(M, D)`, compute the `(N, M)` distance matrix using broadcasting (no loops). Hint: reshape `X` to `(N, 1, D)` and `Y` to `(1, M, D)`, then compute the difference.

4. **Batched outer product.** Given `u` of shape `(B, M)` and `v` of shape `(B, N)`, compute `out[b, i, j] = u[b, i] * v[b, j]` with shape `(B, M, N)` using broadcasting.

5. **Verify with `np.broadcast_shapes`.** Confirm your shape predictions from part 1.

In [ ]:
# YOUR CODE HERE — Exercise 0.2

# 1. Predict shapes (or "error") — fill in your predictions
# (5, 3) + (3,)      →  ?
# (5, 3) + (5,)      →  ?
# (2, 1, 4) + (3, 4) →  ?
# (2, 3, 4) + (2, 1, 1) →  ?
# (7,) + (1, 7, 1)   →  ?

# 2. Normalize images (B, H, W, C) per-channel given mean & std of shape (C,)
# ========================= YOUR CODE HERE =========================

B, H, W, C = 8, 32, 32, 3
# images = rng.standard_normal((B, H, W, C)).astype(np.float32) * 50 + 128
images = rng.standard_normal((B, C, H, W)).astype(np.float32) * 50 + 128
mean = np.array([123.68, 116.78, 103.94], dtype=np.float32)
std = np.array([58.39, 57.12, 57.38], dtype=np.float32)
# normalized = ???

# 3. Pairwise Euclidean distances: X (N, D), Y (M, D) → (N, M)
N, M_pw, D = 100, 80, 5
X = rng.standard_normal((N, D))
Y = rng.standard_normal((M_pw, D))
# dist = ???

# 4. Batched outer product: u (B, M), v (B, N) → (B, M, N)
B_op, M_op, N_op = 16, 5, 7
u = rng.standard_normal((B_op, M_op))
v = rng.standard_normal((B_op, N_op))
# outer = ???


# ========================== END YOUR CODE ==========================

---
## 0.3 — Advanced Indexing: Fancy Indexing, Boolean Masks, `np.where`

NumPy gives you three ways to pull elements out of an array. They look similar on the surface, but behave very differently under the hood — especially around whether you get a view or a copy.

### Basic slicing

This is what you already know: `a[2:5]`, `a[::2]`, `a[:, 1:3]`.

- Returns a **view** — shared memory, zero copy
- Mutating the slice mutates the original
- Always use this when you can; it's fast and free

### Fancy (integer array) indexing

Pass an array of integer indices instead of a slice: `a[[0, 2, 4]]`.

- Always returns a **copy** — a brand new array
- The result shape matches the index array's shape, not the original
- This is how you gather specific elements by position — for example, looking up schedule values at particular timesteps:

```python
# schedule has shape (T,), timesteps has shape (B,)
values_at_t = schedule[timesteps]  # shape (B,) — one value per sample
```

### Boolean mask indexing

Pass a boolean array: `a[a > 0]`, `a[mask]`.

- Also returns a **copy**
- The result is always **1-D**, regardless of the original shape
- Great for filtering: grab all elements meeting some condition

### `np.where` — vectorized if/else

`np.where(condition, x, y)` returns an array where each element comes from `x` (if True) or `y` (if False). Think of it as a vectorized ternary operator — no Python loop needed.

### Other useful functions

- `np.take(a, indices, axis)` — fancy indexing along a specific axis
- `np.take_along_axis(a, indices, axis)` — gather using per-element index arrays (pairs well with `argsort` results)

In [ ]:
# --- Worked Example: 2D fancy indexing ---

grid = np.arange(20).reshape(4, 5)  # shape (4, 5)
print("Grid:")
print(grid)

# Select elements at (0,1), (2,3), (3,4)
row_idx = np.array([0, 2, 3])
col_idx = np.array([1, 3, 4])
selected = grid[row_idx, col_idx]  # shape (3,)
print(f"\nSelected grid[{list(row_idx)}, {list(col_idx)}] = {selected}")

# Boolean mask: replace negatives with 0
data = rng.standard_normal((3, 4))  # shape (3, 4)
print(f"\nOriginal data:\n{data}")
data_clipped = np.where(data > 0, data, 0.0)  # shape (3, 4)
print(f"Clipped (ReLU):\n{data_clipped}")

### Exercise 0.3

**1. Top-k without full sort.**

Given a 1-D array `scores` of shape `(N,)`, find the indices of the top-5 values.

- Use `np.argpartition` (O(N)) instead of `np.argsort` (O(N log N))
- Return the indices sorted in descending order of their values

**2. Batch crop extraction.**

Given `images` of shape `(B, H, W)` and per-image crop coordinates `rows` and `cols`, each of shape `(B,)`, extract the `(B,)` array of pixel values at those coordinates.

- Hint: use `np.arange(B)` for the batch dimension to pair each row/col with the right image

**3. `np.where` from scratch.**

Implement `my_where(cond, x, y)` using only boolean indexing and array allocation (no `np.where`).

- Create an output array, fill it using `cond` and `~cond` as masks
- Verify it matches `np.where` on a test case

In [ ]:
# YOUR CODE HERE — Exercise 0.3

# 1. Top-k without full sort. Use np.argpartition (O(N)) instead of
#    np.argsort (O(N log N)). Return top-5 indices in descending order.
def topk_indices(scores: np.ndarray, k: int = 5) -> np.ndarray:
    """Return indices of top-k values in descending order."""
    # ===================== YOUR CODE HERE =====================
    pass  # Replace this with your implementation
    # ====================== END YOUR CODE ======================

# 2. Batch crop extraction. Given images (B, H, W) and per-image
#    coordinates rows (B,), cols (B,), extract pixels at those coords.
#    Hint: use np.arange for the batch dimension.

# 3. Implement my_where(cond, x, y) using only boolean indexing.
def my_where(cond: np.ndarray, x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Element-wise ternary: return x where cond is True, y otherwise."""
    # ===================== YOUR CODE HERE =====================
    pass  # Replace this with your implementation
    # ====================== END YOUR CODE ======================


---
## 0.4 — Einsum Mastery

Einstein summation (`np.einsum`) gives you a single, readable notation for any tensor contraction — matrix multiplies, dot products, traces, outer products, attention scores, and more.

### The notation

$$
C_{ik} = \sum_j A_{ij} B_{jk} \quad \Longleftrightarrow \quad \texttt{np.einsum('ij,jk->ik', A, B)}
$$

In words: `A` has axes `i, j` and `B` has axes `j, k`. The output keeps `i` and `k`. The shared index `j` appears in both inputs but not in the output, so it gets summed over. That summation is exactly what matrix multiplication does — each element of the result is a dot product along the `j` axis.

### Rules

1. Each input gets a subscript string — one letter per axis.
2. The output subscript specifies which indices survive.
3. Any index in the inputs but **not** in the output gets **summed over** (contracted).

### Common patterns reference

| Operation | Einsum | Equivalent |
|-----------|--------|-----------|
| Matrix multiply | `ij,jk->ik` | `A @ B` |
| Batch matmul | `bij,bjk->bik` | `A @ B` with batch dims |
| Dot product | `i,i->` | `np.dot(a, b)` |
| Outer product | `i,j->ij` | `np.outer(a, b)` |
| Trace | `ii->` | `np.trace(A)` |
| Transpose | `ij->ji` | `A.T` |
| Row sum | `ij->i` | `A.sum(axis=1)` |
| Column sum | `ij->j` | `A.sum(axis=0)` |
| Element-wise multiply then sum | `ij,ij->` | `(A * B).sum()` |
| **Attention scores** | `bhqd,bhkd->bhqk` | $\text{scores}_{b,h,q,k} = \sum_d Q_{b,h,q,d} K_{b,h,k,d}$ |
| **Bilinear form** $\mathbf{x}^\top A \mathbf{x}$ | `i,ij,j->` | Custom |

In [ ]:
# --- Worked Example: Attention scores ---

B_attn, H, Q, K, D_head = 2, 4, 8, 8, 16
queries = rng.standard_normal((B_attn, H, Q, D_head))  # shape (B, H, Q, D)
keys = rng.standard_normal((B_attn, H, K, D_head))     # shape (B, H, K, D)

# Attention: scores[b,h,q,k] = sum_d queries[b,h,q,d] * keys[b,h,k,d]
scores = np.einsum('bhqd,bhkd->bhqk', queries, keys)  # shape (B, H, Q, K)
print(f"Attention scores shape: {scores.shape}")

# Verify against manual loop for one element
manual_val = np.sum(queries[0, 1, 2, :] * keys[0, 1, 3, :])
assert np.isclose(scores[0, 1, 2, 3], manual_val)
print(f"scores[0,1,2,3] = {scores[0,1,2,3]:.4f} (verified)")

In [ ]:
# --- Worked Example: Batched bilinear form x^T A x ---

B_bi, D_bi = 32, 5
x_bi = rng.standard_normal((B_bi, D_bi))           # shape (B, D)
A_bi = rng.standard_normal((B_bi, D_bi, D_bi))     # shape (B, D, D)

# x^T A x for each batch: contract over both i and j
# result[b] = sum_i sum_j x[b,i] * A[b,i,j] * x[b,j]
result_bi = np.einsum('bi,bij,bj->b', x_bi, A_bi, x_bi)  # shape (B,)
print(f"Bilinear form shape: {result_bi.shape}")

# Verify one element
manual_bi = x_bi[0] @ A_bi[0] @ x_bi[0]
assert np.isclose(result_bi[0], manual_bi)
print(f"result[0] = {result_bi[0]:.4f} (verified)")

### Exercise 0.4

**Part A: 10 translations.** For each operation, write the einsum string AND verify it produces the correct result.

1. Matrix-vector product: `A @ v` where `A` is `(M, N)`, `v` is `(N,)`
2. Element-wise product then sum all: `(A * B).sum()`
3. Batch matrix multiply: `X @ Y` where `X` is `(B, M, N)`, `Y` is `(B, N, P)`
4. Trace of a matrix: `np.trace(A)`
5. Diagonal extraction: `np.diag(A)`
6. Sum over last axis: `A.sum(axis=-1)` for 3-D `A`
7. Hadamard (element-wise) product: `A * B` (keep all dims)
8. Batch dot product: `(u * v).sum(axis=-1)` for `u, v` of shape `(B, D)`
9. Tensor contraction: contract middle axis of shape `(A, B, C)` with vector of shape `(B,)` to get `(A, C)`
10. Kronecker-like: outer product of each row of `(M, N)` with itself, producing `(M, N, N)`

**Part B: Multi-head attention dot products.** Given `Q` shape `(B, H, S, D)` and `K` shape `(B, H, S, D)`, compute scaled attention logits $QK^\top / \sqrt{D}$ using einsum.

**Part C: Batch Frobenius norm.** Given `X` shape `(B, M, N)`, compute $\|X_b\|_F$ for each batch element using einsum to get a `(B,)` array of squared norms, then take the square root.

In [ ]:
# YOUR CODE HERE — Exercise 0.4

# Setup: test arrays (don't modify these)
M_a, N_a, P_a, B_a, D_a, C_a = 4, 5, 3, 2, 6, 7
A_sq = rng.standard_normal((N_a, N_a))  # square matrix for trace/diag
A_mn = rng.standard_normal((M_a, N_a))
B_mn = rng.standard_normal((M_a, N_a))
v_n = rng.standard_normal((N_a,))
X_3d = rng.standard_normal((B_a, M_a, N_a))
Y_3d = rng.standard_normal((B_a, N_a, P_a))
u_bd = rng.standard_normal((B_a, D_a))
v_bd = rng.standard_normal((B_a, D_a))
T_abc = rng.standard_normal((M_a, N_a, C_a))
w_b = rng.standard_normal((N_a,))

# Part A: Fill in the einsum strings (replace '???' with the correct subscripts)

# 1. Matrix-vector product: A @ v
r1 = np.einsum('???', A_mn, v_n)
assert np.allclose(r1, A_mn @ v_n), "1 failed"
print("1. mat-vec ✓")

# 2. Element-wise product then sum all: (A * B).sum()
r2 = np.einsum('???', A_mn, B_mn)
assert np.isclose(r2, (A_mn * B_mn).sum()), "2 failed"
print("2. elem-sum ✓")

# 3. Batch matrix multiply: X @ Y
r3 = np.einsum('???', X_3d, Y_3d)
assert np.allclose(r3, X_3d @ Y_3d), "3 failed"
print("3. batch matmul ✓")

# 4. Trace of a matrix: np.trace(A)
r4 = np.einsum('???', A_sq)
assert np.isclose(r4, np.trace(A_sq)), "4 failed"
print("4. trace ✓")

# 5. Diagonal extraction: np.diag(A)
r5 = np.einsum('???', A_sq)
assert np.allclose(r5, np.diag(A_sq)), "5 failed"
print("5. diag ✓")

# 6. Sum over last axis of 3-D tensor: A.sum(axis=-1)
r6 = np.einsum('???', X_3d)
assert np.allclose(r6, X_3d.sum(axis=-1)), "6 failed"
print("6. sum last axis ✓")

# 7. Hadamard (element-wise) product: A * B (keep all dims)
r7 = np.einsum('???', A_mn, B_mn)
assert np.allclose(r7, A_mn * B_mn), "7 failed"
print("7. hadamard ✓")

# 8. Batch dot product: (u * v).sum(axis=-1)
r8 = np.einsum('???', u_bd, v_bd)
assert np.allclose(r8, (u_bd * v_bd).sum(axis=-1)), "8 failed"
print("8. batch dot ✓")

# 9. Contract middle axis of (A,B,C) with vector (B,) → (A,C)
r9 = np.einsum('???', T_abc, w_b)
assert np.allclose(r9, np.tensordot(T_abc, w_b, axes=([1], [0]))), "9 failed"
print("9. contract mid ✓")

# 10. Outer product of each row with itself: (M,N) → (M,N,N)
r10 = np.einsum('???', A_mn, A_mn)
assert np.allclose(r10[0], np.outer(A_mn[0], A_mn[0])), "10 failed"
print("10. row outer ✓")

print("\nAll 10 translations passed!")

# Part B: Multi-head attention logits
# Q shape (B, H, S, D), K shape (B, H, S, D) → scores shape (B, H, S, S)
B_mha, H_mha, S_mha, D_mha = 2, 4, 8, 16
Q_mha = rng.standard_normal((B_mha, H_mha, S_mha, D_mha))
K_mha = rng.standard_normal((B_mha, H_mha, S_mha, D_mha))

# ===================== YOUR CODE HERE =====================
# scores = np.einsum('???', Q_mha, K_mha) / np.sqrt(D_mha)
# ====================== END YOUR CODE ======================

# Part C: Batch Frobenius norm
# X shape (B, M, N) → squared norms shape (B,), then sqrt
X_frob = rng.standard_normal((B_a, M_a, N_a))

# ===================== YOUR CODE HERE =====================
# sq_norms = np.einsum('???', X_frob, X_frob)  # shape (B,)
# frob_norms = np.sqrt(sq_norms)
# ====================== END YOUR CODE ======================


In [ ]:
# ✅ SOLUTION — try the exercise above before running this — Part A: 10 translations

M_a, N_a, P_a, B_a, D_a, C_a = 4, 5, 3, 2, 6, 7
A_sq = rng.standard_normal((N_a, N_a))  # square matrix for trace/diag
A_mn = rng.standard_normal((M_a, N_a))
B_mn = rng.standard_normal((M_a, N_a))
v_n = rng.standard_normal((N_a,))
X_3d = rng.standard_normal((B_a, M_a, N_a))
Y_3d = rng.standard_normal((B_a, N_a, P_a))
u_bd = rng.standard_normal((B_a, D_a))
v_bd = rng.standard_normal((B_a, D_a))
T_abc = rng.standard_normal((M_a, N_a, C_a))
w_b = rng.standard_normal((N_a,))

# 1. Matrix-vector product
r1 = np.einsum('ij,j->i', A_mn, v_n)  # shape (M,)
assert np.allclose(r1, A_mn @ v_n)
print("1. ij,j->i (mat-vec)  OK")

# 2. Element-wise product then sum
r2 = np.einsum('ij,ij->', A_mn, B_mn)  # scalar
assert np.isclose(r2, (A_mn * B_mn).sum())
print("2. ij,ij->  (elem-sum) OK")

# 3. Batch matmul
r3 = np.einsum('bmn,bnp->bmp', X_3d, Y_3d)  # shape (B, M, P)
assert np.allclose(r3, X_3d @ Y_3d)
print("3. bmn,bnp->bmp (batch matmul) OK")

# 4. Trace
r4 = np.einsum('ii->', A_sq)  # scalar
assert np.isclose(r4, np.trace(A_sq))
print("4. ii->  (trace) OK")

# 5. Diagonal
r5 = np.einsum('ii->i', A_sq)  # shape (N,)
assert np.allclose(r5, np.diag(A_sq))
print("5. ii->i (diag) OK")

# 6. Sum over last axis of 3-D
r6 = np.einsum('ijk->ij', X_3d)  # shape (B, M)
assert np.allclose(r6, X_3d.sum(axis=-1))
print("6. ijk->ij (sum last) OK")

# 7. Hadamard product
r7 = np.einsum('ij,ij->ij', A_mn, B_mn)  # shape (M, N)
assert np.allclose(r7, A_mn * B_mn)
print("7. ij,ij->ij (hadamard) OK")

# 8. Batch dot product
r8 = np.einsum('bd,bd->b', u_bd, v_bd)  # shape (B,)
assert np.allclose(r8, (u_bd * v_bd).sum(axis=-1))
print("8. bd,bd->b (batch dot) OK")

# 9. Tensor contraction: contract axis 1 of (A,B,C) with (B,)
r9 = np.einsum('abc,b->ac', T_abc, w_b)  # shape (A, C)
assert np.allclose(r9, np.tensordot(T_abc, w_b, axes=([1], [0])))
print("9. abc,b->ac (contract mid) OK")

# 10. Outer product of each row with itself
r10 = np.einsum('mi,mj->mij', A_mn, A_mn)  # shape (M, N, N)
assert np.allclose(r10[0], np.outer(A_mn[0], A_mn[0]))
print("10. mi,mj->mij (row outer) OK")

---
## 0.5 — Vectorization Patterns: Replacing Loops

The core rule: replace Python `for` loops with NumPy operations. NumPy dispatches to optimized C/Fortran routines that exploit SIMD instructions, cache locality, and parallelism. The performance gap is typically **10–100x**.

If you find yourself iterating over array elements in Python, there is almost certainly a faster way.

### Reductions with `axis`

Functions like `np.sum`, `np.mean`, `np.max`, and `np.min` all accept an `axis` argument. Instead of looping over rows to compute per-row sums, you write `np.sum(x, axis=1)` — one call, no loop.

### Cumulative operations

`np.cumsum` and `np.cumprod` compute running totals and running products along an axis. In DDPM, the noise schedule is defined by $\beta_t$, and the key derived quantity is:

$$
\bar{\alpha}_t = \prod_{s=1}^{t} (1 - \beta_s)
$$

If `betas` is an array of shape `(T,)`, then `alpha_bar = np.cumprod(1 - betas)` gives you a shape `(T,)` array where each element is the running product up to that timestep. No loop needed.

### Sliding windows

`np.lib.stride_tricks.sliding_window_view` extracts overlapping or non-overlapping windows from an array without copying data. Useful for patch extraction (e.g. for Vision Transformers).

### Search and binning

- `np.searchsorted(sorted_array, values)` — binary search into a sorted array
- `np.digitize(data, bin_edges)` — assign each value to a bin
- `np.bincount(indices)` — count occurrences of each integer index

In [ ]:
# --- Worked Example: Cumulative product of alphas (diffusion!) ---
# In DDPM, we define:
#   beta_t: noise schedule
#   alpha_t = 1 - beta_t
#   alpha_bar_t = prod_{s=1}^{t} alpha_s  (cumulative product)

T_diff = 1000
betas = np.linspace(1e-4, 0.02, T_diff, dtype=np.float64)  # shape (T,)
alphas = 1.0 - betas                                        # shape (T,)
alpha_bar = np.cumprod(alphas)                               # shape (T,)

print(f"betas range: [{betas[0]:.4f}, {betas[-1]:.4f}]")
print(f"alpha_bar[0]   = {alpha_bar[0]:.6f}  (close to 1: minimal noise)")
print(f"alpha_bar[-1]  = {alpha_bar[-1]:.6f}  (close to 0: mostly noise)")
print(f"alpha_bar[500] = {alpha_bar[500]:.6f}  (midway)")

In [ ]:
# --- Worked Example: Vectorized histogram ---

data_hist = rng.standard_normal(10000)  # shape (10000,)
bin_edges = np.array([-3, -2, -1, 0, 1, 2, 3], dtype=np.float64)

# Method 1: np.digitize + np.bincount
bin_indices = np.digitize(data_hist, bin_edges)  # shape (10000,) values in [0, len(edges)]
counts = np.bincount(bin_indices, minlength=len(bin_edges) + 1)  # shape (8,)
print("Manual histogram (digitize+bincount):")
for i in range(len(bin_edges) - 1):
    print(f"  [{bin_edges[i]:+.0f}, {bin_edges[i+1]:+.0f}): {counts[i+1]}")

# Method 2: np.histogram (built-in)
counts2, _ = np.histogram(data_hist, bins=bin_edges)
print(f"\nnp.histogram counts: {counts2}")

### Exercise 0.5

**1. Loop to vectorized: row norms.**

Replace the loop with a single vectorized expression.

```python
X = rng.standard_normal((1000, 50))
norms = np.zeros(1000)
for i in range(1000):
    norms[i] = np.sqrt(np.sum(X[i] ** 2))
```

- Hint: `np.sum` and `np.sqrt` both accept an `axis` argument

**2. Cumulative max.**

Compute the running maximum of a 1-D array without a loop.

- Hint: `np.maximum.accumulate`

**3. Pairwise absolute differences.**

Given `a` of shape `(N,)`, compute the `(N, N)` matrix where entry `[i, j]` is `|a_i - a_j|`.

- Use broadcasting — no loops

**4. Cumprod via cumsum + exp/log.**

Implement cumulative product using only `np.cumsum`, `np.log`, and `np.exp`.

- Compare numerical precision against `np.cumprod` on the alpha schedule

**5. Timed pairwise cosine similarity.**

For `X` of shape `(500, 128)`, compute the `(500, 500)` cosine similarity matrix.

- Time both a loop version and a vectorized version to see the speedup

In [ ]:
# YOUR CODE HERE — Exercise 0.5

# 1. Row norms — vectorized (replace the loop)
X_norm = rng.standard_normal((1000, 50))  # shape (1000, 50)

# Loop version (for reference — this is what you're replacing)
norms_loop = np.zeros(1000)
for i in range(1000):
    norms_loop[i] = np.sqrt(np.sum(X_norm[i] ** 2))

# ===================== YOUR CODE HERE =====================
norms_vec = None  # Replace with a single vectorized expression
# ====================== END YOUR CODE ======================

assert norms_vec is not None, "Fill in norms_vec"
assert np.allclose(norms_loop, norms_vec), f"Mismatch: loop vs vectorized"
print("1. Row norms ✓")


# 2. Cumulative max (no loops)
arr_cm = rng.standard_normal(20)  # shape (20,)

# ===================== YOUR CODE HERE =====================
cummax = None  # Replace with a single expression
# ====================== END YOUR CODE ======================

assert cummax is not None, "Fill in cummax"
# Verify: each element should be >= all previous elements
assert all(cummax[i] >= cummax[i-1] for i in range(1, len(cummax)))
assert cummax[-1] == arr_cm.max()
print("2. Cumulative max ✓")


# 3. Pairwise absolute differences — no loops, use broadcasting
a_pw = rng.standard_normal(100)  # shape (100,)

# ===================== YOUR CODE HERE =====================
pairwise_abs = None  # Replace: should have shape (100, 100)
# ====================== END YOUR CODE ======================

assert pairwise_abs is not None, "Fill in pairwise_abs"
assert pairwise_abs.shape == (100, 100)
assert np.isclose(pairwise_abs[10, 20], abs(a_pw[10] - a_pw[20]))
assert np.allclose(pairwise_abs, pairwise_abs.T)  # should be symmetric
print("3. Pairwise abs diff ✓")


# 4. Cumprod via cumsum + exp/log (only valid for positive arrays)
def cumprod_via_cumsum(arr: np.ndarray) -> np.ndarray:
    """Cumulative product using log-space: exp(cumsum(log(arr)))."""
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================

test_arr = np.array([0.999, 0.998, 0.997, 0.996, 0.995])
result = cumprod_via_cumsum(test_arr)
assert result is not None, "Fill in cumprod_via_cumsum"
assert np.allclose(result, np.cumprod(test_arr), atol=1e-10)
print("4. Cumprod via cumsum ✓")


# 5. Timed pairwise cosine similarity
X_cos = rng.standard_normal((500, 128))  # shape (500, 128)

# ===================== YOUR CODE HERE =====================
# Compute the (500, 500) cosine similarity matrix.
# Cosine similarity: cos(a, b) = (a · b) / (||a|| * ||b||)
# Hint: normalize rows first, then a single matrix multiply does it.
cos_vec = None  # Replace with your vectorized implementation
# ====================== END YOUR CODE ======================

assert cos_vec is not None, "Fill in cos_vec"
assert cos_vec.shape == (500, 500)
# Diagonal should be 1.0 (each vector with itself)
assert np.allclose(np.diag(cos_vec), 1.0, atol=1e-6)
print("5. Cosine similarity ✓")

print("\nAll exercises passed!")


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# 1. Row norms -- vectorized
X_norm = rng.standard_normal((1000, 50))  # shape (1000, 50)

# Loop version (slow)
norms_loop = np.zeros(1000)
for i in range(1000):
    norms_loop[i] = np.sqrt(np.sum(X_norm[i] ** 2))

# Vectorized version
norms_vec = np.sqrt(np.sum(X_norm ** 2, axis=1))  # shape (1000,)
# Alternative: np.linalg.norm(X_norm, axis=1)

assert np.allclose(norms_loop, norms_vec)
print("1. Row norms match.")

# 2. Cumulative max
arr_cm = rng.standard_normal(20)  # shape (20,)
cummax = np.maximum.accumulate(arr_cm)  # shape (20,)
print(f"\n2. Cumulative max:")
print(f"   Input:  {arr_cm[:8]}")
print(f"   CumMax: {cummax[:8]}")

# Verify
cummax_loop = np.zeros_like(arr_cm)
cummax_loop[0] = arr_cm[0]
for i in range(1, len(arr_cm)):
    cummax_loop[i] = max(cummax_loop[i-1], arr_cm[i])
assert np.allclose(cummax, cummax_loop)

# 3. Pairwise absolute differences
a_pw = rng.standard_normal(100)  # shape (100,)
pairwise_abs = np.abs(a_pw[:, None] - a_pw[None, :])  # shape (100, 100)
print(f"\n3. Pairwise abs diff shape: {pairwise_abs.shape}")
assert pairwise_abs[10, 20] == abs(a_pw[10] - a_pw[20])
print("   Verified.")

# 4. Cumprod via cumsum + exp/log
def cumprod_via_cumsum(arr: np.ndarray) -> np.ndarray:
    """Cumulative product using log-space: exp(cumsum(log(arr))).
    
    Only valid for strictly positive arrays.
    """
    return np.exp(np.cumsum(np.log(arr)))


# Test on alpha schedule (all values in (0,1), so log is valid)
alpha_bar_logspace = cumprod_via_cumsum(alphas)  # shape (T,)
max_err = np.max(np.abs(alpha_bar_logspace - alpha_bar))
print(f"\n4. Max error (log-space cumprod vs np.cumprod): {max_err:.2e}")
assert np.allclose(alpha_bar_logspace, alpha_bar, atol=1e-10)

# 5. Timed pairwise cosine similarity
X_cos = rng.standard_normal((500, 128))  # shape (500, 128)

# Loop version
t0 = time.perf_counter()
norms_x = np.linalg.norm(X_cos, axis=1)  # precompute for fair comparison
cos_loop = np.zeros((500, 500))
for i in range(500):
    for j in range(500):
        cos_loop[i, j] = np.dot(X_cos[i], X_cos[j]) / (norms_x[i] * norms_x[j])
t_loop = time.perf_counter() - t0

# Vectorized version
t0 = time.perf_counter()
X_normed = X_cos / np.linalg.norm(X_cos, axis=1, keepdims=True)  # shape (500, 128)
cos_vec = X_normed @ X_normed.T  # shape (500, 500)
t_vec = time.perf_counter() - t0

print(f"\n5. Cosine similarity:")
print(f"   Loop version:       {t_loop*1000:.1f} ms")
print(f"   Vectorized version: {t_vec*1000:.1f} ms")
print(f"   Speedup: {t_loop/t_vec:.0f}x")
assert np.allclose(cos_loop, cos_vec, atol=1e-10)

---
## 0.6 — Random Number Generation

A quick note on the right way to generate random numbers in NumPy.

### Use `np.random.default_rng(seed)`, not the legacy API

The old `np.random.seed()` / `np.random.randn()` API relies on global state, which is fragile and hard to reason about. The modern `Generator` API is what you should use:

```python
rng = np.random.default_rng(42)
noise = rng.standard_normal((B, C, H, W))
```

Each `Generator` has its own state, the same seed always produces the same sequence, and there's no global state to accidentally corrupt. We'll use this pattern throughout the course.

In [ ]:
# --- Worked Example: Reproducible sampling ---

rng1 = np.random.default_rng(seed=123)
rng2 = np.random.default_rng(seed=123)

samples1 = rng1.standard_normal((3, 4))  # shape (3, 4)
samples2 = rng2.standard_normal((3, 4))  # shape (3, 4)

print("Same seed -> identical samples:")
print(f"  samples match: {np.array_equal(samples1, samples2)}")

# Different seed -> different samples
rng3 = np.random.default_rng(seed=456)
samples3 = rng3.standard_normal((3, 4))  # shape (3, 4)
print(f"  different seed matches: {np.array_equal(samples1, samples3)}")

### Exercise 0.6

**1. Reproducibility check.**

Create two `Generator` objects with the same seed. Draw `(5, 5)` samples from each and verify they're identical.

**2. Seeded noise for a batch of images.**

Given a batch shape `(B, C, H, W) = (4, 3, 32, 32)`, generate noise using `rng.standard_normal`. Verify the mean is close to 0 and std is close to 1.

In [ ]:
# YOUR CODE HERE — Exercise 0.6

# 1. Reproducibility check
# ===================== YOUR CODE HERE =====================
# Create two Generators with the same seed, draw (5, 5) samples, verify identical

# ====================== END YOUR CODE ======================
print("1. Reproducibility ✓")


# 2. Seeded noise for a batch of images
# ===================== YOUR CODE HERE =====================
# Generate noise of shape (4, 3, 32, 32) using rng.standard_normal
# Verify mean ≈ 0 and std ≈ 1

# ====================== END YOUR CODE ======================
print("2. Batch noise ✓")


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# 1. Reproducibility check
rng_a = np.random.default_rng(seed=99)
rng_b = np.random.default_rng(seed=99)
samples_a = rng_a.standard_normal((5, 5))
samples_b = rng_b.standard_normal((5, 5))
assert np.array_equal(samples_a, samples_b), "Should be identical!"
print("1. Same seed → identical samples ✓")

# 2. Seeded noise for a batch of images
rng_img = np.random.default_rng(42)
noise = rng_img.standard_normal((4, 3, 32, 32))  # (B, C, H, W)
print(f"2. Noise shape: {noise.shape}")
print(f"   Mean: {noise.mean():.4f} (expected ~0)")
print(f"   Std:  {noise.std():.4f} (expected ~1)")

---
## Capstone Challenges

Five exercises that integrate everything from this module. Each has a solution cell immediately following.

### Challenge 1: Batch Normalization from Scratch

Implement **batch normalization** for a tensor of shape `(B, C, H, W)`:

```
mu_c = mean over (B, H, W) for each channel
var_c = var over (B, H, W) for each channel
x_hat = (x - mu_c) / sqrt(var_c + eps)
out = gamma * x_hat + beta
```

Here `gamma` and `beta` are learnable per-channel parameters of shape `(C,)`. Each channel gets its own scale and shift, applied element-wise after normalization.

To make the shapes work, you will need to reshape `gamma` and `beta` to `(1, C, 1, 1)` so they broadcast correctly over the batch, height, and width dimensions.

In [ ]:
# YOUR CODE HERE — Challenge 1

def batch_norm(
    x: np.ndarray,
    gamma: np.ndarray,
    beta: np.ndarray,
    eps: float = 1e-5
) -> np.ndarray:
    """Batch normalization for (B, C, H, W) tensors.
    
    Steps:
    1. Compute per-channel mean and variance over (B, H, W)
    2. Normalize: x_hat = (x - mu) / sqrt(var + eps)
    3. Scale and shift: out = gamma * x_hat + beta
    
    Hint: use keepdims=True and reshape gamma/beta to (1, C, 1, 1).
    
    Expected: per-channel mean ≈ beta, std ≈ gamma after normalization.
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace this with your implementation
    # ====================== END YOUR CODE ======================


In [ ]:
# ✅ SOLUTION — Challenge 1

def batch_norm(
    x: np.ndarray,
    gamma: np.ndarray,
    beta: np.ndarray,
    eps: float = 1e-5
) -> np.ndarray:
    """Batch normalization for (B, C, H, W) tensors."""
    # Per-channel mean and variance over (B, H, W)
    mu = x.mean(axis=(0, 2, 3), keepdims=True)        # shape (1, C, 1, 1)
    var = x.var(axis=(0, 2, 3), keepdims=True)         # shape (1, C, 1, 1)
    
    # Normalize
    x_hat = (x - mu) / np.sqrt(var + eps)              # shape (B, C, H, W)
    
    # Scale and shift
    gamma_r = gamma.reshape(1, -1, 1, 1)               # shape (1, C, 1, 1)
    beta_r = beta.reshape(1, -1, 1, 1)                 # shape (1, C, 1, 1)
    return gamma_r * x_hat + beta_r                     # shape (B, C, H, W)


# Test
rng_bn = np.random.default_rng(42)
x_bn = rng_bn.standard_normal((8, 3, 16, 16)) * 5 + 10  # non-zero mean, large std
gamma_bn = np.ones(3)
beta_bn = np.zeros(3)

out_bn = batch_norm(x_bn, gamma_bn, beta_bn)
channel_means = out_bn.mean(axis=(0, 2, 3))
channel_stds = out_bn.std(axis=(0, 2, 3))
print(f"Per-channel means: {channel_means.round(6)} (expected ~0)")
print(f"Per-channel stds:  {channel_stds.round(4)} (expected ~1)")
assert np.allclose(channel_means, 0, atol=1e-6)
assert np.allclose(channel_stds, 1, atol=0.01)
print("Batch norm verified ✓")


### Challenge 2: Numerically Stable Softmax

Implement softmax along the last axis with the **max-subtraction trick** for numerical stability.

The naive formula `exp(x) / sum(exp(x))` overflows when `x` contains large values (try `x = [1000, 1001, 1002]`). Subtracting the max before exponentiation fixes this:

$$
\text{softmax}(x)_i = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}
$$

The result is mathematically identical but each exponent is now at most 0, so no overflow. Verify your implementation handles extreme values without NaN/Inf.

In [ ]:
# YOUR CODE HERE — Challenge 2

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax along the given axis.
    
    Use the max-subtraction trick to prevent overflow.
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Tests — these should all pass when your implementation is correct
rng_sm = np.random.default_rng(42)

# Test 1: basic functionality
logits = rng_sm.standard_normal((4, 10))
probs = softmax(logits)
assert probs is not None, "softmax returned None"
assert np.allclose(probs.sum(axis=1), 1.0), f"Rows don't sum to 1: {probs.sum(axis=1)}"
assert (probs > 0).all(), "All probabilities should be positive"
print("Test 1 (basic): ✓")

# Test 2: extreme values (this is the whole point of max-subtraction)
logits_extreme = np.array([[1000.0, 1001.0, 1002.0]])
probs_extreme = softmax(logits_extreme)
assert not np.any(np.isnan(probs_extreme)), "Got NaN on extreme values!"
assert not np.any(np.isinf(probs_extreme)), "Got Inf on extreme values!"
print("Test 2 (extreme values): ✓")

# Test 3: works on 3-D tensors
logits_3d = rng_sm.standard_normal((2, 3, 5))
probs_3d = softmax(logits_3d, axis=-1)
assert np.allclose(probs_3d.sum(axis=-1), 1.0), "3-D softmax failed"
print("Test 3 (3-D tensor): ✓")


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax along the given axis.
    
    Subtracts the max before exponentiation to prevent overflow.
    """
    x_max = x.max(axis=axis, keepdims=True)  # shape: x with axis kept
    e_x = np.exp(x - x_max)                  # shape: same as x
    return e_x / e_x.sum(axis=axis, keepdims=True)  # shape: same as x


# Test: normal values
rng_sm = np.random.default_rng(42)
logits = rng_sm.standard_normal((4, 10))  # shape (4, 10)
probs = softmax(logits)  # shape (4, 10)
print(f"Sum per row (should be 1.0): {probs.sum(axis=1)}")
print(f"All positive: {(probs > 0).all()}")

# Test: extreme values (would cause overflow without max trick)
logits_extreme = np.array([[1000.0, 1001.0, 1002.0],
                           [-1000.0, -999.0, -998.0]])  # shape (2, 3)
probs_extreme = softmax(logits_extreme)  # shape (2, 3)
print(f"\nExtreme logits softmax:\n{probs_extreme}")
print(f"No NaN: {not np.any(np.isnan(probs_extreme))}")
print(f"No Inf: {not np.any(np.isinf(probs_extreme))}")
print(f"Sums: {probs_extreme.sum(axis=1)}")

# Test: 3-D tensor
logits_3d = rng_sm.standard_normal((2, 3, 5))  # shape (2, 3, 5)
probs_3d = softmax(logits_3d, axis=-1)  # shape (2, 3, 5)
print(f"\n3-D softmax sums: {probs_3d.sum(axis=-1)}")

### Challenge 3: KNN — Pairwise Distances + Top-k Vectorized

Implement **k-nearest neighbors** classification, fully vectorized:

1. Compute pairwise L2 distances between query points `(Q, D)` and training points `(N, D)` using the expansion $\|q - x\|^2 = \|q\|^2 + \|x\|^2 - 2 q \cdot x$, which avoids an expensive `(Q, N, D)` intermediate.
2. Find the k nearest neighbors using `np.argpartition` (O(N) per query, faster than sorting).
3. Return the majority label via `np.bincount`.

In [ ]:
# YOUR CODE HERE — Challenge 3

def knn_predict(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_query: np.ndarray,
    k: int = 5
) -> np.ndarray:
    """K-nearest neighbors classification, fully vectorized.
    
    Args:
        X_train: Training features, shape (N, D).
        y_train: Training labels, shape (N,), integer class labels.
        X_query: Query features, shape (Q, D).
        k: Number of neighbors.
    
    Returns:
        Predicted labels, shape (Q,).
    
    Hints:
        - ||q - x||^2 = ||q||^2 + ||x||^2 - 2(q · x)  (avoids (Q,N,D) intermediate)
        - np.argpartition for top-k (faster than full sort)
        - np.bincount for majority vote
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Test with synthetic clustered data
rng_knn = np.random.default_rng(42)
N_tr, D_knn, n_classes_knn = 200, 2, 3
centers = np.array([[0, 0], [5, 5], [0, 5]], dtype=np.float64)
X_tr = np.vstack([
    rng_knn.standard_normal((N_tr // n_classes_knn, D_knn)) + centers[c]
    for c in range(n_classes_knn)
])
y_tr = np.repeat(np.arange(n_classes_knn), N_tr // n_classes_knn)

# Query near each center — should predict the correct class
X_q = centers + rng_knn.standard_normal((3, 2)) * 0.1
preds = knn_predict(X_tr, y_tr, X_q, k=5)
assert preds is not None, "knn_predict returned None"
assert np.array_equal(preds, [0, 1, 2]), f"Expected [0, 1, 2], got {preds}"
print(f"Predictions: {preds} ✓")


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def knn_predict(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_query: np.ndarray,
    k: int = 5
) -> np.ndarray:
    """K-nearest neighbors classification, fully vectorized.
    
    Args:
        X_train: Training features, shape (N, D).
        y_train: Training labels, shape (N,), integer class labels.
        X_query: Query features, shape (Q, D).
        k: Number of neighbors.
    
    Returns:
        Predicted labels, shape (Q,).
    """
    # Pairwise L2 distances: (Q, N)
    # ||q - x||^2 = ||q||^2 + ||x||^2 - 2 q.x  (more memory-efficient than broadcasting diff)
    q_sq = np.sum(X_query ** 2, axis=1, keepdims=True)  # shape (Q, 1)
    x_sq = np.sum(X_train ** 2, axis=1, keepdims=True)  # shape (N, 1)
    dist_sq = q_sq + x_sq.T - 2.0 * X_query @ X_train.T  # shape (Q, N)
    dist_sq = np.maximum(dist_sq, 0.0)  # numerical safety
    
    # Top-k nearest (smallest distances) using argpartition
    knn_idx = np.argpartition(dist_sq, k, axis=1)[:, :k]  # shape (Q, k)
    
    # Gather labels of k nearest neighbors
    knn_labels = y_train[knn_idx]  # shape (Q, k)
    
    # Majority vote per query
    n_classes = y_train.max() + 1
    predictions = np.zeros(X_query.shape[0], dtype=y_train.dtype)  # shape (Q,)
    for q in range(X_query.shape[0]):
        counts = np.bincount(knn_labels[q], minlength=n_classes)  # shape (n_classes,)
        predictions[q] = counts.argmax()
    
    return predictions


# Test with synthetic data
rng_knn = np.random.default_rng(42)
N_tr, D_knn, n_classes_knn = 200, 2, 3

# Generate clustered data
centers = np.array([[0, 0], [5, 5], [0, 5]], dtype=np.float64)  # shape (3, 2)
X_tr = np.vstack([
    rng_knn.standard_normal((N_tr // n_classes_knn, D_knn)) + centers[c]
    for c in range(n_classes_knn)
])  # shape (N_tr, 2) -- approximately, due to integer division
y_tr = np.repeat(np.arange(n_classes_knn), N_tr // n_classes_knn)  # shape (N_tr,)

# Query near each center
X_q = centers + rng_knn.standard_normal((3, 2)) * 0.1  # shape (3, 2)
preds = knn_predict(X_tr, y_tr, X_q, k=5)  # shape (3,)
print(f"Query points near centers 0, 1, 2")
print(f"Predictions: {preds}  (expected: [0, 1, 2])")
assert np.array_equal(preds, [0, 1, 2])

### Challenge 4: Image Patch Extraction with reshape/strides

Extract non-overlapping patches of size `(P, P)` from an image of shape `(H, W)` using only `reshape` and `transpose` (no loops, no `sliding_window_view`). The result should have shape `(H//P, W//P, P, P)`.

The key insight: reshape the image to `(H//P, P, W//P, P)`, then transpose axes 1 and 2 to group the patch dimensions together. This is exactly how Vision Transformers (ViT) "patchify" their input images.

In [ ]:
# YOUR CODE HERE — Challenge 4

def extract_patches(image: np.ndarray, patch_size: int) -> np.ndarray:
    """Extract non-overlapping patches from a 2-D image.
    
    Args:
        image: Shape (H, W), where H and W are divisible by patch_size.
        patch_size: Side length of square patches.
    
    Returns:
        Patches of shape (H//P, W//P, P, P).
    
    Hint: reshape to (H//P, P, W//P, P), then transpose axes 1 and 2.
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Tests
img = np.arange(96).reshape(8, 12)  # shape (8, 12)
patches = extract_patches(img, 4)
assert patches is not None, "extract_patches returned None"
assert patches.shape == (2, 3, 4, 4), f"Expected (2, 3, 4, 4), got {patches.shape}"
assert np.array_equal(patches[0, 0], img[:4, :4]), "Top-left patch mismatch"
assert np.array_equal(patches[1, 2], img[4:8, 8:12]), "Bottom-right patch mismatch"
print(f"Patches shape: {patches.shape} ✓")
print("All patches verified ✓")


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def extract_patches(image: np.ndarray, patch_size: int) -> np.ndarray:
    """Extract non-overlapping patches from a 2-D image.
    
    Args:
        image: Shape (H, W), where H and W are divisible by patch_size.
        patch_size: Side length of square patches.
    
    Returns:
        Patches of shape (H//P, W//P, P, P).
    """
    H, W = image.shape
    P = patch_size
    assert H % P == 0 and W % P == 0, "Dimensions must be divisible by patch_size"
    
    # Reshape: (H, W) -> (H//P, P, W//P, P) -> transpose to (H//P, W//P, P, P)
    patches = image.reshape(H // P, P, W // P, P)  # shape (H//P, P, W//P, P)
    patches = patches.transpose(0, 2, 1, 3)         # shape (H//P, W//P, P, P)
    return patches


# Test
H_p, W_p, P_size = 8, 12, 4
img = np.arange(H_p * W_p).reshape(H_p, W_p)  # shape (8, 12)
patches = extract_patches(img, P_size)  # shape (2, 3, 4, 4)
print(f"Image shape: {img.shape}")
print(f"Patches shape: {patches.shape}")
print(f"\nPatch [0, 0]:")
print(patches[0, 0])
print(f"\nTop-left 4x4 of image:")
print(img[:4, :4])
assert np.array_equal(patches[0, 0], img[:4, :4])
assert np.array_equal(patches[1, 2], img[4:8, 8:12])
print("\nAll patches verified.")

### Challenge 5: Convolution via im2col

Instead of sliding a kernel across an image with nested loops, deep learning frameworks use the **im2col** trick:

1. **im2col:** Extract every overlapping patch into rows of a matrix (using `sliding_window_view`).
2. **Matrix multiply:** Multiply the patch matrix by the flattened kernel in one shot.

This reduces convolution to a highly optimized GEMM (general matrix multiply) call, which is how cuDNN and MKL-DNN actually implement it.

**Implement** 2-D convolution (no padding, stride=1):

- Input: `image` shape `(H, W)`, `kernel` shape `(kH, kW)`
- Output: shape `(H-kH+1, W-kW+1)`
- Use `sliding_window_view` to extract patches with shape `(oH, oW, kH, kW)`
- Reshape to `(oH*oW, kH*kW)` and matrix-multiply with `kernel.reshape(-1)`
- Verify against a manual check: `sum(image[r:r+kH, c:c+kW] * kernel)` at one position

In [ ]:
# YOUR CODE HERE — Challenge 5

from numpy.lib.stride_tricks import sliding_window_view

def conv2d_im2col(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """2-D convolution via im2col (no padding, stride=1).
    
    Args:
        image: Shape (H, W).
        kernel: Shape (kH, kW).
    
    Returns:
        Output of shape (H-kH+1, W-kW+1).
    
    Steps:
        1. Use sliding_window_view to extract patches: (oH, oW, kH, kW)
        2. Reshape to (oH*oW, kH*kW)
        3. Matrix-multiply with flattened kernel
        4. Reshape back to (oH, oW)
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Tests
rng_conv = np.random.default_rng(42)
img_conv = rng_conv.standard_normal((10, 10))
kernel_conv = np.array([[1, 0, -1],
                        [2, 0, -2],
                        [1, 0, -1]], dtype=np.float64)  # Sobel-x

out = conv2d_im2col(img_conv, kernel_conv)
assert out is not None, "conv2d_im2col returned None"
assert out.shape == (8, 8), f"Expected (8, 8), got {out.shape}"

# Verify against manual convolution at one position
manual_val = np.sum(img_conv[2:5, 3:6] * kernel_conv)
assert np.isclose(out[2, 3], manual_val), f"Mismatch at [2,3]: {out[2,3]} vs {manual_val}"
print(f"Output shape: {out.shape} ✓")
print(f"Spot check [2,3]: {out[2,3]:.4f} ✓")


In [ ]:
# ✅ SOLUTION — try the exercise above before running this

from numpy.lib.stride_tricks import sliding_window_view as _swv


def conv2d_im2col(
    image: np.ndarray, kernel: np.ndarray
) -> np.ndarray:
    """2-D convolution via im2col (no padding, stride=1).
    
    Args:
        image: Shape (H, W).
        kernel: Shape (kH, kW).
    
    Returns:
        Output of shape (H-kH+1, W-kW+1).
    """
    kH, kW = kernel.shape
    # Extract all overlapping patches: shape (oH, oW, kH, kW)
    patches = _swv(image, (kH, kW))  # shape (oH, oW, kH, kW)
    oH, oW = patches.shape[0], patches.shape[1]
    
    # Flatten patches and kernel for matrix multiply
    col_matrix = patches.reshape(oH * oW, kH * kW)  # shape (oH*oW, kH*kW)
    kernel_flat = kernel.reshape(-1)                  # shape (kH*kW,)
    
    # Matrix-vector product
    output_flat = col_matrix @ kernel_flat  # shape (oH*oW,)
    return output_flat.reshape(oH, oW)     # shape (oH, oW)


# Test
rng_conv = np.random.default_rng(42)
img_conv = rng_conv.standard_normal((10, 10))  # shape (10, 10)
kernel_conv = np.array([[1, 0, -1],
                        [2, 0, -2],
                        [1, 0, -1]], dtype=np.float64)  # shape (3, 3) -- Sobel-x

out_conv = conv2d_im2col(img_conv, kernel_conv)  # shape (8, 8)
print(f"Input shape: {img_conv.shape}")
print(f"Kernel shape: {kernel_conv.shape}")
print(f"Output shape: {out_conv.shape}")

# Verify against manual convolution at one position
manual_val = np.sum(img_conv[2:5, 3:6] * kernel_conv)
assert np.isclose(out_conv[2, 3], manual_val)
print(f"out[2,3] = {out_conv[2,3]:.4f} (verified against manual: {manual_val:.4f})")

### Challenge 6: Cumulative Alpha Schedule

Given a beta schedule of shape `(T,)`, compute all the derived quantities used in DDPM:

- `alpha_t = 1 - beta_t` — shape `(T,)`
- `alpha_bar_t = prod_{s=1}^{t} alpha_s` — shape `(T,)`, the cumulative product
- `sqrt_alpha_bar_t` — shape `(T,)`
- `sqrt_one_minus_alpha_bar_t` — shape `(T,)`
- `posterior_variance_t = beta_t * (1 - alpha_bar_{t-1}) / (1 - alpha_bar_t)` — shape `(T,)`, with the convention that `alpha_bar_0 = 1.0`

These quantities appear constantly in the diffusion forward process, training loss, and reverse sampling.

In [ ]:
# YOUR CODE HERE — Challenge 6

def compute_schedule(betas: np.ndarray) -> dict:
    """Compute all derived DDPM schedule quantities from a beta schedule.
    
    Args:
        betas: Shape (T,), noise schedule values.
    
    Returns:
        Dict with keys:
            'alphas': shape (T,)
            'alpha_bar': shape (T,)
            'sqrt_alpha_bar': shape (T,)
            'sqrt_one_minus_alpha_bar': shape (T,)
            'posterior_variance': shape (T,), using alpha_bar_0 = 1.0
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Replace with your implementation
    # ====================== END YOUR CODE ======================


# Tests
T_test = 1000
betas_test = np.linspace(1e-4, 0.02, T_test, dtype=np.float64)
schedule = compute_schedule(betas_test)

assert schedule is not None, "compute_schedule returned None"
assert schedule['alphas'].shape == (T_test,)
assert schedule['alpha_bar'].shape == (T_test,)
assert np.isclose(schedule['alpha_bar'][0], 1 - betas_test[0]), "alpha_bar[0] wrong"
assert schedule['alpha_bar'][-1] < 0.01, "alpha_bar should decay to near 0"
assert np.allclose(schedule['sqrt_alpha_bar'] ** 2, schedule['alpha_bar'])
assert np.allclose(schedule['sqrt_one_minus_alpha_bar'] ** 2, 1 - schedule['alpha_bar'])
assert schedule['posterior_variance'].shape == (T_test,)
print(f"alpha_bar[0] = {schedule['alpha_bar'][0]:.6f} (near 1) ✓")
print(f"alpha_bar[-1] = {schedule['alpha_bar'][-1]:.6f} (near 0) ✓")
print("All schedule quantities verified ✓")


In [ ]:
# ✅ SOLUTION — Challenge 6

def compute_schedule(betas: np.ndarray) -> dict:
    alphas = 1.0 - betas                                    # shape (T,)
    alpha_bar = np.cumprod(alphas)                           # shape (T,)
    sqrt_alpha_bar = np.sqrt(alpha_bar)                      # shape (T,)
    sqrt_one_minus_alpha_bar = np.sqrt(1.0 - alpha_bar)      # shape (T,)
    
    # Posterior variance: beta_t * (1 - alpha_bar_{t-1}) / (1 - alpha_bar_t)
    alpha_bar_prev = np.concatenate([[1.0], alpha_bar[:-1]])  # shape (T,), with alpha_bar_0 = 1
    posterior_variance = betas * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar)  # shape (T,)
    
    return {
        'alphas': alphas,
        'alpha_bar': alpha_bar,
        'sqrt_alpha_bar': sqrt_alpha_bar,
        'sqrt_one_minus_alpha_bar': sqrt_one_minus_alpha_bar,
        'posterior_variance': posterior_variance,
    }

# Verify
T_test = 1000
betas_test = np.linspace(1e-4, 0.02, T_test, dtype=np.float64)
s = compute_schedule(betas_test)
print(f"alpha_bar: {s['alpha_bar'][0]:.4f} → {s['alpha_bar'][-1]:.6f}")
print(f"sqrt_alpha_bar: {s['sqrt_alpha_bar'][0]:.4f} → {s['sqrt_alpha_bar'][-1]:.4f}")
print(f"posterior_var[1]: {s['posterior_variance'][1]:.6f}")


---

**Module 0 complete.** You now have the NumPy fluency needed to implement diffusion models from scratch.

Key takeaways:

- Understand memory layout (strides, views vs. copies) to avoid silent bugs and unnecessary allocations
- Broadcasting eliminates loops and temporary arrays — master the three rules
- Einsum is a universal tool for any tensor contraction; learn the notation once, use it everywhere
- Always vectorize: the performance gap between Python loops and NumPy operations is typically 10-100x
- Use the modern `Generator` API for reproducible random numbers
- The reparameterization trick ($z = \mu + \sigma \cdot \varepsilon$) is foundational to diffusion and VAE sampling

Proceed to **Module 1** when ready.